# Repository guide: MMS full training; no augmentation

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


# MMS-1B → Tarifit V1.2 Adapter Fine-Tuning — No Augmentation

**Goal:** train the first final V1.2 ASR system using `facebook/mms-1b-all` with a fresh Tarifit CTC vocabulary and adapter layers.

This notebook deliberately:
- uses only the frozen **train + validation** portion of Corpus V1.2;
- excludes every row assigned to `test`;
- uses the final 31-letter Tarifit alphabet;
- disables SpecAugment for this **no-augmentation baseline**;
- trains only the MMS adapter parameters and CTC output layer;
- uses validation WER/CER for checkpoint selection;
- saves a small best-adapter file instead of multi-GB full-model checkpoints;
- does **not** evaluate the test set.

A second notebook/experiment can later repeat the same setup with augmentation.

In [ ]:
# Cell 1 — Install reproducible dependencies
!pip -q install "transformers==4.57.1" "datasets==4.4.1" "accelerate>=1.10,<2" "jiwer==4.0.0" "safetensors>=0.4.5" "soundfile>=0.12.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# Cell 2 — Mount Google Drive and define experiment paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2.csv"
FROZEN_METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2_train_val_frozen.csv"

TOKENIZER_DIR = PROJECT_ROOT / "data" / "processed" / "mms_tokenizer_v1_2"
DATASET_CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_2"
CACHE_MANIFEST_PATH = DATASET_CACHE_DIR / "cache_manifest.json"

MODEL_DIR = PROJECT_ROOT / "models" / "mms_1b_tarifit_v1_2_noaug"
RESULTS_DIR = PROJECT_ROOT / "results" / "mms_1b_tarifit_v1_2_noaug"

BEST_ADAPTER_PATH = MODEL_DIR / "best_adapter.safetensors"
BEST_ADAPTER_INFO_PATH = MODEL_DIR / "best_adapter_info.json"

for path in [TOKENIZER_DIR, MODEL_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Metadata:", METADATA_PATH)
print("Model output:", MODEL_DIR)
print("Results:", RESULTS_DIR)

Mounted at /content/drive
Project root: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm
Metadata: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/segments_metadata_v1_2.csv
Model output: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_noaug
Results: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_noaug


In [ ]:
# Cell 3 — Record software and GPU environment
import sys
import torch
import transformers
import datasets
import jiwer
import soundfile

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU memory (GB):", round(props.total_memory / 1024**3, 2))

Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 4.57.1
Datasets: 4.4.1
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


In [ ]:
# Cell 4 — Load V1.2 metadata and select final train/validation rows

import pandas as pd
import numpy as np

df = pd.read_csv(METADATA_PATH)

df["transcription"] = (
    df["transcription"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["final_selection"] = (
    df["final_selection"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

df["review_status"] = (
    df["review_status"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

df["dataset_split"] = (
    df["dataset_split"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)


# Train: selected + included + transcription available
train_selected = df[
    df["dataset_split"].eq("train")
    & df["final_selection"].eq("yes")
    & df["review_status"].isin(["reviewed", "included"])
    & df["transcription"].ne("")
].copy()


# Validation: selected + reviewed + transcription available
val_selected = df[
    df["dataset_split"].eq("validation")
    & df["final_selection"].eq("yes")
    & df["review_status"].eq("reviewed")
    & df["transcription"].ne("")
].copy()


selected = pd.concat(
    [train_selected, val_selected],
    ignore_index=True
)

assert len(train_selected) > 0, "No training rows found."
assert len(val_selected) > 0, "No validation rows found."
assert not selected["segment_id"].duplicated().any(), "Duplicate segment_id values found."

print("Selected rows:", len(selected))

print("\nRows by split:")
print(selected["dataset_split"].value_counts())

print("\nHours by split:")
print(
    selected.groupby("dataset_split")["duration_seconds"]
    .sum()
    .div(3600)
    .round(3)
)

print("\nSpeakers by split:")
print(
    selected.groupby("dataset_split")["speaker_group_id"]
    .nunique()
)

Selected rows: 1883

Rows by split:
dataset_split
train         1754
validation     129
Name: count, dtype: int64

Hours by split:
dataset_split
train         5.223
validation    0.298
Name: duration_seconds, dtype: float64

Speakers by split:
dataset_split
train         3
validation    2
Name: speaker_group_id, dtype: int64


In [ ]:
# Cell 4B — Inspect all V1.2 training-row statuses

train_all = df[
    df["dataset_split"].eq("train")
].copy()

print("All train rows:", len(train_all))

print("\nFinal selection:")
print(train_all["final_selection"].value_counts(dropna=False))

print("\nReview status:")
print(train_all["review_status"].value_counts(dropna=False))

print("\nTranscription available:")
print(
    train_all["transcription"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .value_counts()
)

print("\nCross-tab final_selection × review_status:")
print(
    pd.crosstab(
        train_all["final_selection"],
        train_all["review_status"],
        dropna=False
    )
)

All train rows: 2756

Final selection:
final_selection
yes    1754
no     1002
Name: count, dtype: int64

Review status:
review_status
reviewed    1472
included    1272
rejected      12
Name: count, dtype: int64

Transcription available:
transcription
True     1754
False    1002
Name: count, dtype: int64

Cross-tab final_selection × review_status:
review_status    included  rejected  reviewed
final_selection                              
no                    990        12         0
yes                   282         0      1472


In [ ]:
# Cell 4C — Remove the obsolete cache created from the incorrect 282-segment train set

import shutil

if FROZEN_METADATA_PATH.exists():
    FROZEN_METADATA_PATH.unlink()
    print("Removed old frozen metadata.")

if DATASET_CACHE_DIR.exists():
    shutil.rmtree(DATASET_CACHE_DIR)
    print("Removed old V1.2 dataset cache.")

print("✓ Ready to rebuild with the correct 1,754-segment train set.")

Removed old frozen metadata.
Removed old V1.2 dataset cache.
✓ Ready to rebuild with the correct 1,754-segment train set.


In [ ]:
# Cell 5 — Verify speaker independence and complete test exclusion
train_rows = selected[selected["dataset_split"] == "train"].copy()
val_rows = selected[selected["dataset_split"] == "validation"].copy()

train_speakers = set(train_rows["speaker_group_id"].dropna())
val_speakers = set(val_rows["speaker_group_id"].dropna())

test_rows_all = df[df["dataset_split"].eq("test")].copy()
test_speakers = set(test_rows_all["speaker_group_id"].dropna())

train_val_overlap = train_speakers & val_speakers
train_test_overlap = train_speakers & test_speakers
val_test_overlap = val_speakers & test_speakers

print("Train speakers:", sorted(train_speakers))
print("Validation speakers:", sorted(val_speakers))
print("Reserved test speakers:", sorted(test_speakers))
print()
print("Train ∩ validation:", train_val_overlap)
print("Train ∩ test:", train_test_overlap)
print("Validation ∩ test:", val_test_overlap)

assert not train_val_overlap, "Speaker leakage between train and validation."
assert not train_test_overlap, "Speaker leakage between train and reserved test data."
assert not val_test_overlap, "Speaker leakage between validation and reserved test data."

assert not selected["dataset_split"].eq("test").any(), "Test rows entered train/validation selection."

print("\n✓ No speaker leakage and no test rows in the training experiment.")

Train speakers: ['SPK001', 'SPK002', 'SPK009']
Validation speakers: ['SPK007', 'SPK010']
Reserved test speakers: ['SPK003', 'SPK004', 'SPK006', 'SPK008']

Train ∩ validation: set()
Train ∩ test: set()
Validation ∩ test: set()

✓ No speaker leakage and no test rows in the training experiment.


In [ ]:
# Cell 6 — Validate final V1.2 orthographic character inventory
from collections import Counter
import unicodedata

FINAL_LETTERS = list("abcdefghijklmnpqrstuvwxyz") + ["ɛ", "ɣ", "ʷ", "ḍ", "ḥ", "ṭ"]
ALLOWED_CHARACTERS = set(FINAL_LETTERS) | {" "}

char_counts = Counter("".join(selected["transcription"].tolist()))
unexpected = {
    ch: count
    for ch, count in char_counts.items()
    if ch not in ALLOWED_CHARACTERS
}

print("Expected final letters:", " ".join(FINAL_LETTERS))
print("Number of expected letters:", len(FINAL_LETTERS))
print("Unexpected characters:", unexpected)

for text in selected["transcription"]:
    assert text == unicodedata.normalize("NFC", text), "Non-NFC transcription found."

assert not unexpected, f"Unexpected transcription characters remain: {unexpected}"

print("✓ V1.2 train/validation text matches the final normalized alphabet.")

Expected final letters: a b c d e f g h i j k l m n p q r s t u v w x y z ɛ ɣ ʷ ḍ ḥ ṭ
Number of expected letters: 31
Unexpected characters: {}
✓ V1.2 train/validation text matches the final normalized alphabet.


In [ ]:
# Cell 7 — Freeze the exact V1.2 train/validation metadata used by this experiment
import hashlib

freeze_columns = [
    "segment_id",
    "recording_id",
    "speaker_group_id",
    "dataset_split",
    "duration_seconds",
    "audio_path",
    "review_status",
    "transcription",
    "final_selection",
]

current_frozen = (
    selected[freeze_columns]
    .sort_values(["dataset_split", "segment_id"])
    .reset_index(drop=True)
)

if FROZEN_METADATA_PATH.exists():
    existing_frozen = (
        pd.read_csv(FROZEN_METADATA_PATH)
        [freeze_columns]
        .sort_values(["dataset_split", "segment_id"])
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        current_frozen,
        existing_frozen,
        check_dtype=False,
    )
    print("✓ Existing frozen train/validation metadata matches V1.2.")
else:
    current_frozen.to_csv(
        FROZEN_METADATA_PATH,
        index=False,
        encoding="utf-8",
    )
    print("✓ Frozen train/validation metadata created.")

sha256 = hashlib.sha256(FROZEN_METADATA_PATH.read_bytes()).hexdigest()

print("Frozen metadata:", FROZEN_METADATA_PATH)
print("SHA256:", sha256)

✓ Existing frozen train/validation metadata matches V1.2.
Frozen metadata: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/segments_metadata_v1_2_train_val_frozen.csv
SHA256: 4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab


## Tokenizer decision

The final orthography is already defined, so the tokenizer is **not learned from validation text**.  
We construct it deterministically from the fixed 31-letter alphabet.

For CTC:
- space is represented internally by `|`;
- `[PAD]` is also the CTC blank token;
- `[UNK]` is retained as a safety token;
- no BOS/EOS tokens are needed.

In [ ]:
# Cell 8 — Build and save the deterministic V1.2 CTC tokenizer
import json
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

vocab_tokens = ["|"] + FINAL_LETTERS + ["[UNK]", "[PAD]"]
vocab_dict = {token: idx for idx, token in enumerate(vocab_tokens)}

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"

with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
)

BASE_MODEL_ID = "facebook/mms-1b-all"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    BASE_MODEL_ID,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

processor.save_pretrained(TOKENIZER_DIR)

print("Tokenizer size:", len(tokenizer))
print("PAD/CTC blank id:", tokenizer.pad_token_id)
print("UNK id:", tokenizer.unk_token_id)
print("\nVocabulary:")
print(tokenizer.get_vocab())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

Tokenizer size: 34
PAD/CTC blank id: 33
UNK id: 32

Vocabulary:
{'|': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, 'ɛ': 26, 'ɣ': 27, 'ʷ': 28, 'ḍ': 29, 'ḥ': 30, 'ṭ': 31, '[UNK]': 32, '[PAD]': 33}


In [ ]:
# Cell 9 — Verify every selected transcription encodes without UNK
unk_id = tokenizer.unk_token_id
unknown_segments = []

for row in current_frozen.itertuples(index=False):
    ids = tokenizer(row.transcription).input_ids
    if unk_id in ids:
        unknown_segments.append((row.segment_id, row.transcription))

print("Segments containing [UNK]:", len(unknown_segments))

if unknown_segments:
    for item in unknown_segments[:20]:
        print(item)

assert not unknown_segments, "Some V1.2 references contain characters outside the tokenizer."

print("✓ All train/validation references are fully representable.")

Segments containing [UNK]: 0
✓ All train/validation references are fully representable.


## Dataset preparation

The WAV files are already standardized to **16 kHz mono** in the project.  
For each selected segment we create:
- `input_values`: waveform values consumed by MMS;
- `input_length`: number of audio samples, useful for integrity/CTC checks;
- `labels`: character-level CTC target IDs.

The processed dataset is cached on Drive. On reruns, the notebook reuses the cache only if the frozen metadata hash is unchanged.

In [ ]:
# Cell 10 — Build or reload the cached V1.2 Hugging Face dataset
from datasets import Dataset, DatasetDict, load_from_disk
import soundfile as sf

def prepare_split(frame):
    work = frame.copy()
    work["absolute_audio_path"] = work["audio_path"].apply(
        lambda p: str(PROJECT_ROOT / p)
    )
    return Dataset.from_pandas(
        work[
            [
                "segment_id",
                "recording_id",
                "speaker_group_id",
                "duration_seconds",
                "absolute_audio_path",
                "transcription",
            ]
        ],
        preserve_index=False,
    )

def prepare_example(example):
    audio, sr = sf.read(
        example["absolute_audio_path"],
        dtype="float32",
        always_2d=False,
    )

    if sr != 16000:
        raise ValueError(
            f'{example["segment_id"]}: expected 16000 Hz, found {sr} Hz'
        )

    if getattr(audio, "ndim", 1) != 1:
        raise ValueError(
            f'{example["segment_id"]}: audio is not mono'
        )

    model_inputs = processor(
        audio,
        sampling_rate=16000,
    )

    labels = tokenizer(
        example["transcription"]
    ).input_ids

    return {
        "input_values": model_inputs.input_values[0],
        "input_length": len(model_inputs.input_values[0]),
        "labels": labels,
    }

use_existing_cache = False

if DATASET_CACHE_DIR.exists() and CACHE_MANIFEST_PATH.exists():
    with open(CACHE_MANIFEST_PATH, "r", encoding="utf-8") as f:
        cache_manifest = json.load(f)

    if cache_manifest.get("metadata_sha256") == sha256:
        use_existing_cache = True

if use_existing_cache:
    dataset = load_from_disk(str(DATASET_CACHE_DIR))
    print("✓ Reused existing V1.2 dataset cache.")
else:
    if DATASET_CACHE_DIR.exists():
        raise RuntimeError(
            "A V1.2 cache exists but its metadata hash does not match the frozen corpus. "
            "Do not overwrite it silently; inspect/remove the stale cache intentionally."
        )

    raw_dataset = DatasetDict({
        "train": prepare_split(
            current_frozen[current_frozen["dataset_split"] == "train"]
        ),
        "validation": prepare_split(
            current_frozen[current_frozen["dataset_split"] == "validation"]
        ),
    })

    dataset = raw_dataset.map(
        prepare_example,
        remove_columns=[
            "absolute_audio_path",
            "transcription",
            "recording_id",
            "speaker_group_id",
            "duration_seconds",
        ],
        desc="Preparing 16 kHz audio and CTC labels",
    )

    DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    dataset.save_to_disk(str(DATASET_CACHE_DIR))

    with open(CACHE_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "metadata_sha256": sha256,
                "tokenizer_size": len(tokenizer),
                "base_model": BASE_MODEL_ID,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("✓ Built and saved V1.2 dataset cache.")

train_ds = dataset["train"]
val_ds = dataset["validation"]

print("\nTrain examples:", len(train_ds))
print("Validation examples:", len(val_ds))
print("Dataset features:", train_ds.column_names)

✓ Reused existing V1.2 dataset cache.

Train examples: 1754
Validation examples: 129
Dataset features: ['segment_id', 'input_values', 'input_length', 'labels']


In [ ]:
# Cell 11 — Summarize processed train/validation durations
def duration_summary(split_ds):
    seconds = np.array(split_ds["input_length"], dtype=np.float64) / 16000.0
    return {
        "segments": len(seconds),
        "hours": float(seconds.sum() / 3600),
        "min_s": float(seconds.min()),
        "max_s": float(seconds.max()),
        "mean_s": float(seconds.mean()),
    }

print("Train:", duration_summary(train_ds))
print("Validation:", duration_summary(val_ds))

Train: {'segments': 1754, 'hours': 5.222733593749999, 'min_s': 0.848, 'max_s': 19.984, 'mean_s': 10.71940760404789}
Validation: {'segments': 129, 'hours': 0.2983577777777778, 'min_s': 0.944, 'max_s': 26.0, 'mean_s': 8.326263565891473}


## Model initialization

This experiment uses the official `facebook/mms-1b-all` checkpoint.

We create a **new V1.2 CTC output vocabulary**, reinitialize the MMS adapter layers for Tarifit, freeze the large multilingual base model, and train only the small adapter/CTC parameters. This is the memory-efficient low-resource MMS adaptation strategy.

Because this is the **no-augmentation experiment**, model-level SpecAugment is explicitly disabled.

In [ ]:
# Cell 12 — Load MMS-1B with the V1.2 CTC head and initialize trainable adapters
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_ID,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    apply_spec_augment=False,
    mask_time_prob=0.0,
    mask_feature_prob=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=len(tokenizer),
    ignore_mismatched_sizes=True,
)

# Reinitialize the language-specific MMS adapter/CTC parameters for Tarifit.
model.init_adapter_layers()

# Freeze the shared 1B multilingual base.
model.freeze_base_model()

# Explicitly enable gradients only for adapter parameters.
adapter_weights = model._get_adapters()
for parameter in adapter_weights.values():
    parameter.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")
print("SpecAugment enabled:", model.config.apply_spec_augment)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/mms-1b-all were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.masked_spec_embed']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([34]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([34, 1280]) in the model instantiated
Y

Total parameters: 964,690,850
Trainable parameters: 2,194,722
Trainable percentage: 0.2275%
SpecAugment enabled: False


## CTC feasibility

CTC needs enough output time steps to express the target sequence.

For a target with repeated adjacent characters, CTC needs an extra blank between repetitions. Therefore:

`minimum required CTC frames = label length + number of adjacent repeated labels`

The notebook computes the model's exact output-frame length and **stops before training** if any train or validation pair is impossible.

In [ ]:
# Cell 13 — Run exact CTC feasibility checks on train and validation
def minimum_ctc_frames(labels):
    repeats = sum(
        labels[i] == labels[i - 1]
        for i in range(1, len(labels))
    )
    return len(labels) + repeats

def find_ctc_infeasible(split_ds):
    bad = []

    for i, example in enumerate(split_ds):
        input_samples = int(example["input_length"])

        output_frames = int(
            model._get_feat_extract_output_lengths(
                torch.tensor(input_samples)
            ).item()
        )

        labels = example["labels"]
        min_frames = minimum_ctc_frames(labels)

        if output_frames < min_frames:
            bad.append({
                "index": i,
                "segment_id": example["segment_id"],
                "input_samples": input_samples,
                "audio_seconds": input_samples / 16000.0,
                "output_frames": output_frames,
                "label_length": len(labels),
                "minimum_ctc_frames": min_frames,
            })

    return bad

bad_train = find_ctc_infeasible(train_ds)
bad_val = find_ctc_infeasible(val_ds)

print("CTC-infeasible training examples:", len(bad_train))
print("CTC-infeasible validation examples:", len(bad_val))

if bad_train:
    print("\nTraining problems:")
    display(pd.DataFrame(bad_train))

if bad_val:
    print("\nValidation problems:")
    display(pd.DataFrame(bad_val))

assert not bad_train, "Training contains CTC-infeasible examples. Inspect before training."
assert not bad_val, "Validation contains CTC-infeasible examples. Inspect before training."

print("\n✓ All train and validation examples are CTC-feasible.")

CTC-infeasible training examples: 0
CTC-infeasible validation examples: 0

✓ All train and validation examples are CTC-feasible.


In [ ]:
# Cell 14 — Define dynamic CTC padding for audio and labels
from dataclasses import dataclass
from typing import Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True,
)

print("✓ CTC data collator ready.")

✓ CTC data collator ready.


In [ ]:
# Cell 15 — Define validation WER and CER
from jiwer import wer, cer

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids)
    label_str = tokenizer.batch_decode(
        label_ids,
        group_tokens=False,
    )

    return {
        "wer": wer(label_str, pred_str),
        "cer": cer(label_str, pred_str),
    }

print("✓ WER/CER metric function ready.")

✓ WER/CER metric function ready.


## Training configuration

The default configuration is deliberately conservative for a Colab T4:

- **4 epochs**
- batch size **2**
- gradient accumulation **8** → effective batch size 16
- learning rate **1e-3**
- linear scheduler
- 50 warm-up steps
- FP16
- gradient checkpointing
- validation after every epoch

No full 1B checkpoints are written. A callback saves only the **best adapter weights** according to validation CER.

In [ ]:
# Cell 16 — Set seeds and define the best-adapter checkpoint callback
import random
from transformers import TrainerCallback
from safetensors.torch import save_file as safe_save_file

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

class BestAdapterCallback(TrainerCallback):
    def __init__(self, adapter_path, info_path):
        self.adapter_path = Path(adapter_path)
        self.info_path = Path(info_path)
        self.best_cer = float("inf")

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        metrics = metrics or {}
        current_cer = metrics.get("eval_cer")

        if current_cer is None or model is None:
            return control

        if current_cer < self.best_cer:
            self.best_cer = float(current_cer)

            adapter_state = {
                name: tensor.detach().cpu().contiguous()
                for name, tensor in model._get_adapters().items()
            }

            safe_save_file(
                adapter_state,
                str(self.adapter_path),
                metadata={"format": "pt"},
            )

            info = {
                "best_cer": self.best_cer,
                "eval_wer": float(metrics.get("eval_wer", float("nan"))),
                "eval_loss": float(metrics.get("eval_loss", float("nan"))),
                "epoch": float(state.epoch) if state.epoch is not None else None,
                "global_step": int(state.global_step),
                "metadata_sha256": sha256,
                "base_model": BASE_MODEL_ID,
                "tokenizer_size": len(tokenizer),
            }

            with open(self.info_path, "w", encoding="utf-8") as f:
                json.dump(info, f, indent=2)

            print(
                f"\n✓ New best adapter saved — CER={self.best_cer:.4f}, "
                f"step={state.global_step}"
            )

        return control

best_adapter_callback = BestAdapterCallback(
    BEST_ADAPTER_PATH,
    BEST_ADAPTER_INFO_PATH,
)

print("✓ Reproducibility seed:", SEED)

✓ Reproducibility seed: 42


In [ ]:
# Cell 17 — Configure resumable no-augmentation training

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "trainer_state"),
    num_train_epochs=4,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=1e-3,
    warmup_steps=50,
    lr_scheduler_type="linear",

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",

    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    logging_strategy="steps",
    logging_steps=25,

    report_to="none",
    seed=SEED,
    data_seed=SEED,

    dataloader_num_workers=2,
    remove_unused_columns=True,
)

print("✓ TrainingArguments created.")
print("Epochs:", training_args.num_train_epochs)
print("Save strategy:", training_args.save_strategy)
print("Evaluation strategy:", training_args.eval_strategy)

✓ TrainingArguments created.
Epochs: 4
Save strategy: SaveStrategy.EPOCH
Evaluation strategy: IntervalStrategy.EPOCH


In [ ]:
# Cell 18 — Create the Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor.feature_extractor,
    callbacks=[best_adapter_callback],
)

print("✓ Trainer ready.")

✓ Trainer ready.


In [ ]:
# Cell 19 — Train the V1.2 no-augmentation MMS adapter
train_result = trainer.train()

print("\nTraining complete.")
print(train_result)
print("\nBest adapter path:", BEST_ADAPTER_PATH)
print("Best adapter info:", BEST_ADAPTER_INFO_PATH)

Epoch,Training Loss,Validation Loss,Wer,Cer
1,2.633500,3.580611,0.982143,0.487981
2,0.666800,3.095321,0.881378,0.442319
3,0.587500,3.058462,0.866497,0.434199
4,0.570100,3.112741,0.875425,0.437334



✓ New best adapter saved — CER=0.4880, step=110

✓ New best adapter saved — CER=0.4423, step=220

✓ New best adapter saved — CER=0.4342, step=330

Training complete.
TrainOutput(global_step=440, training_loss=1.5860361207615246, metrics={'train_runtime': 3869.7091, 'train_samples_per_second': 1.813, 'train_steps_per_second': 0.114, 'total_flos': 8.583373530856172e+18, 'train_loss': 1.5860361207615246, 'epoch': 4.0})

Best adapter path: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_noaug/best_adapter.safetensors
Best adapter info: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_noaug/best_adapter_info.json


In [ ]:
# Cell 20 — Reload the best validation-CER adapter
from safetensors.torch import load_file as safe_load_file

assert BEST_ADAPTER_PATH.exists(), "Best adapter was not saved."

best_adapter_state = safe_load_file(
    str(BEST_ADAPTER_PATH),
    device="cpu",
)

load_result = model.load_state_dict(
    best_adapter_state,
    strict=False,
)

print("Adapter tensors loaded:", len(best_adapter_state))
print("Unexpected keys:", len(load_result.unexpected_keys))
print("Missing keys are expected because the frozen base model is not in the adapter file.")

assert len(load_result.unexpected_keys) == 0, (
    f"Unexpected adapter keys: {load_result.unexpected_keys[:20]}"
)

model.to(training_args.device)

with open(BEST_ADAPTER_INFO_PATH, "r", encoding="utf-8") as f:
    best_info = json.load(f)

print("\nBest checkpoint information:")
print(json.dumps(best_info, indent=2))

Adapter tensors loaded: 290
Unexpected keys: 0
Missing keys are expected because the frozen base model is not in the adapter file.

Best checkpoint information:
{
  "best_cer": 0.43419889058606,
  "eval_wer": 0.8664965986394558,
  "eval_loss": 3.058462381362915,
  "epoch": 3.0,
  "global_step": 330,
  "metadata_sha256": "4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab",
  "base_model": "facebook/mms-1b-all",
  "tokenizer_size": 34
}


In [ ]:
# Cell 21 — Evaluate the best adapter on validation
validation_metrics = trainer.evaluate(
    eval_dataset=val_ds,
    metric_key_prefix="validation",
)

print("Best-adapter validation metrics:")
for key, value in validation_metrics.items():
    print(f"{key}: {value}")

Best-adapter validation metrics:
validation_loss: 3.058462381362915
validation_wer: 0.8664965986394558
validation_cer: 0.43419889058606
validation_runtime: 22.2001
validation_samples_per_second: 5.811
validation_steps_per_second: 2.928
epoch: 4.0


In [ ]:
# Cell 22 — Save validation predictions for qualitative error analysis
prediction_output = trainer.predict(
    val_ds,
    metric_key_prefix="validation_prediction",
)

pred_ids = np.argmax(
    prediction_output.predictions,
    axis=-1,
)

pred_texts = tokenizer.batch_decode(pred_ids)

reference_texts = [
    tokenizer.decode(
        example["labels"],
        group_tokens=False,
    )
    for example in val_ds
]

validation_predictions = pd.DataFrame({
    "segment_id": val_ds["segment_id"],
    "reference": reference_texts,
    "prediction": pred_texts,
})

PREDICTIONS_PATH = RESULTS_DIR / "validation_predictions.csv"

validation_predictions.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8",
)

display(validation_predictions.head(20))

print("\nSaved:", PREDICTIONS_PATH)

,segment_id,reference,prediction
0,REC090_SEG0010,ssalamuɛlikum necc meryem,salamuɛlikum nec meryam
1,REC090_SEG0011,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,aqay ruxa tnayn uɛecrin sanadi hulanḍa
2,REC090_SEG0012,mercex ak nmis n jjiran usiɣ ḍ zi lmeɣrib umi ...,mercex ag misenjjiran usiɣ dzi lmeɣrib umiraqq...
3,REC090_SEG0013,umi wsiɣd dda ufix manayenni wa dji ca min ira...,umiwsiɣ dda ufix manayenni wadji ca mirira ɣar...
4,REC090_SEG0014,a necc mammec ira djjix ḍi lmeɣrib wadji manay...,necc mamci ra dji xti lmeɣrib wadji manayniufi...
5,REC090_SEG0015,necc ḍi lmeɣrib ira ɣari lḥurriya inu ira ɣari...,necdi lmeɣrib ira ɣari lḥurriya inu ila ɣari i...
6,REC090_SEG0016,ḍi lmeɣrib neccin mammec ira niɛicc,ḍi lmaɣrim nccin mamciraniɛic
7,REC090_SEG0017,ak baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥway...,ag babadimma waɣanex ca ɣanx caneḥwayj nteggit...
8,REC090_SEG0018,lmuhim wsiɣd,muhimma usiɣt
9,REC090_SEG0019,necc ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ḍi ṭi...,neccira ɛemmas wawsiɣ d ɣa uruppa wsiɣ ddi tiy...



Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_noaug/validation_predictions.csv


In [ ]:
# Cell 23 — Save training history and experiment summary
history_df = pd.DataFrame(trainer.state.log_history)
HISTORY_PATH = RESULTS_DIR / "training_history.csv"
history_df.to_csv(HISTORY_PATH, index=False)

processor.save_pretrained(MODEL_DIR / "processor")
model.config.save_pretrained(MODEL_DIR / "config")

summary = {
    "experiment": "MMS-1B Tarifit V1.2 adapter fine-tuning — no augmentation",
    "base_model": BASE_MODEL_ID,
    "metadata_path": str(FROZEN_METADATA_PATH),
    "metadata_sha256": sha256,
    "train_segments": len(train_ds),
    "validation_segments": len(val_ds),
    "train_hours": duration_summary(train_ds)["hours"],
    "validation_hours": duration_summary(val_ds)["hours"],
    "tokenizer_size": len(tokenizer),
    "final_letters": FINAL_LETTERS,
    "seed": SEED,
    "num_train_epochs": 4,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "effective_train_batch_size": 16,
    "learning_rate": 1e-3,
    "warmup_steps": 50,
    "augmentation": "none; apply_spec_augment=False",
    "best_adapter": best_info,
    "final_validation_metrics": {
        key: float(value) if isinstance(value, (int, float, np.floating)) else value
        for key, value in validation_metrics.items()
    },
}

SUMMARY_PATH = RESULTS_DIR / "experiment_summary.json"

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved training history:", HISTORY_PATH)
print("Saved experiment summary:", SUMMARY_PATH)
print("Saved processor:", MODEL_DIR / "processor")
print("Saved best adapter:", BEST_ADAPTER_PATH)

Saved training history: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_noaug/training_history.csv
Saved experiment summary: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_1b_tarifit_v1_2_noaug/experiment_summary.json
Saved processor: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_noaug/processor
Saved best adapter: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/mms_1b_tarifit_v1_2_noaug/best_adapter.safetensors


## What happens after this notebook?

1. Keep this run as the **V1.2 no-augmentation baseline**.
2. Inspect validation WER/CER and prediction examples.
3. Create the controlled **V1.2 + augmentation** experiment using the same:
   - frozen train/validation metadata,
   - tokenizer,
   - base model,
   - training seed,
   - training hyperparameters,
   - validation set.
4. Finish and freeze the manually reviewed held-out test references.
5. Select final model(s) using **validation only**.
6. Evaluate the selected model(s) once on the held-out test set.

The gap between WER and CER is not arbitrary. A substantial part of the model's error comes from word-boundary errors, orthographic detail, and gemination, while the underlying character sequence is often considerably closer to the reference.